# Forecast Combination V1

This notebook combines multiple imperfect equity signals into a more robust forecast. It uses free price data from `yfinance` and evaluates whether combined forecasts rank future one-month returns better than individual signals.

The important research discipline here is not finding a magic signal. It is learning how to diagnose signals, avoid look-ahead bias, shrink unstable weights, and evaluate forecasts with walk-forward testing.

## 1. Imports and Configuration

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TICKERS = [
    'AAPL', 'MSFT', 'NVDA', 'AMZN', 'META', 'GOOGL', 'TSLA', 'AVGO', 'JPM', 'V',
    'MA', 'UNH', 'HD', 'WMT', 'PG', 'JNJ', 'XOM', 'LLY', 'BAC', 'KO',
    'PEP', 'ADBE', 'CRM', 'NFLX', 'AMD', 'INTC', 'CSCO', 'ORCL', 'MCD', 'DIS',
    'NKE', 'PFE', 'MRK', 'CVX', 'ABBV', 'WFC', 'GS', 'MS', 'BA', 'CAT'
]

START_DATE = '2016-01-01'
TRAIN_MONTHS = 36
RIDGE_ALPHA = 10.0

print(f'Project root: {ROOT}')
print(f'Universe size: {len(TICKERS)}')

## 2. Download Prices and Build Monthly Panel

Signals are observed at month-end and used to forecast the next month's return. This one-period shift is the main look-ahead-bias control in this prototype.

In [ ]:
prices = yf.download(TICKERS + ['SPY'], start=START_DATE, auto_adjust=True, progress=False, threads=True)

if isinstance(prices.columns, pd.MultiIndex):
    close = prices['Close'].copy()
    volume = prices['Volume'].copy()
else:
    close = prices[['Close']].copy()
    volume = prices[['Volume']].copy()

monthly_close = close.resample('ME').last()
monthly_volume = volume.resample('ME').mean()
monthly_returns = monthly_close.pct_change()
market_returns = monthly_returns['SPY'].rename('market_return')

raw_panel = monthly_returns.drop(columns=['SPY'], errors='ignore').stack().rename('return_1m').reset_index()
raw_panel.columns = ['date', 'ticker', 'return_1m']
raw_panel.to_csv(RAW_DIR / 'forecast_combination_monthly_returns.csv', index=False)

print(raw_panel.shape)
raw_panel.head()

## 3. Base Signal Library

The signals are deliberately simple and diversified by construction: momentum, short-term reversal, volatility, beta, liquidity, and distance from highs. Every signal is lagged so that it only uses information known at the forecast date.

In [ ]:
def rolling_beta(asset_returns: pd.Series, market: pd.Series, window: int = 12) -> pd.Series:
    cov = asset_returns.rolling(window).cov(market)
    var = market.rolling(window).var()
    return cov / var


signal_frames = []
for ticker in TICKERS:
    if ticker not in monthly_returns.columns:
        continue
    r = monthly_returns[ticker]
    c = monthly_close[ticker]
    v = monthly_volume[ticker]
    m = market_returns.reindex(r.index)

    df = pd.DataFrame({
        'date': r.index,
        'ticker': ticker,
        'target_next_return': r.shift(-1),
        'momentum_12_1': c.pct_change(12).shift(1),
        'momentum_6_1': c.pct_change(6).shift(1),
        'reversal_1m': -r,
        'low_volatility': -r.rolling(12).std(),
        'market_beta': -rolling_beta(r, m, 12),
        'liquidity': np.log(v.replace(0, np.nan)),
        'distance_to_12m_high': (c / c.rolling(12).max()) - 1,
    })
    signal_frames.append(df)

panel = pd.concat(signal_frames, ignore_index=True)
signal_cols = ['momentum_12_1', 'momentum_6_1', 'reversal_1m', 'low_volatility', 'market_beta', 'liquidity', 'distance_to_12m_high']

# Cross-sectional z-scores keep signal scales comparable and reduce one-name dominance.
for col in signal_cols:
    panel[col] = panel.groupby('date')[col].transform(lambda s: (s - s.mean()) / s.std(ddof=0))

panel = panel.replace([np.inf, -np.inf], np.nan).dropna(subset=['target_next_return']).reset_index(drop=True)
panel.to_csv(PROCESSED_DIR / 'forecast_combination_panel_v1.csv', index=False)

print(panel.shape)
panel.head()

## 4. Signal Diagnostics

IC means information coefficient: the cross-sectional Spearman rank correlation between a signal and next-period returns. A signal can be weak each month but useful if the average IC is positive and persistent.

In [ ]:
def monthly_ic(data: pd.DataFrame, signal: str) -> pd.Series:
    return data.groupby('date').apply(
        lambda g: g[signal].corr(g['target_next_return'], method='spearman') if g[[signal, 'target_next_return']].dropna().shape[0] >= 8 else np.nan
    )


ic_table = pd.DataFrame({sig: monthly_ic(panel, sig) for sig in signal_cols})
ic_summary = pd.DataFrame({
    'mean_ic': ic_table.mean(),
    'ic_vol': ic_table.std(),
    'ic_ir': ic_table.mean() / ic_table.std(),
}).sort_values('mean_ic', ascending=False)
display(ic_summary)

plt.figure(figsize=(10, 5))
ic_table.rolling(6).mean().plot(ax=plt.gca())
plt.title('Six-Month Rolling IC by Base Signal')
plt.ylabel('Spearman IC')
plt.show()

## 5. Walk-Forward Combination Methods

In [ ]:
def fit_predict_walk_forward(data: pd.DataFrame) -> pd.DataFrame:
    dates = sorted(data['date'].dropna().unique())
    predictions = []

    for i in range(TRAIN_MONTHS, len(dates) - 1):
        train_dates = dates[i - TRAIN_MONTHS:i]
        test_date = dates[i]
        train = data[data['date'].isin(train_dates)].copy()
        test = data[data['date'] == test_date].copy()

        X_train = train[signal_cols]
        y_train = train['target_next_return']
        X_test = test[signal_cols]

        equal_weights = np.repeat(1 / len(signal_cols), len(signal_cols))
        ic_window = pd.DataFrame({sig: monthly_ic(train, sig) for sig in signal_cols})
        inv_vol = 1 / ic_window.std().replace(0, np.nan)
        inv_vol_weights = (inv_vol / inv_vol.sum()).fillna(1 / len(signal_cols)).values
        bma_scores = np.exp(6 * ic_window.mean().fillna(0))
        bma_weights = (bma_scores / bma_scores.sum()).values

        models = {
            'ols': Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', LinearRegression())]),
            'ridge': Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', Ridge(alpha=RIDGE_ALPHA))]),
        }

        out = test[['date', 'ticker', 'target_next_return']].copy()
        out['equal_weight'] = np.nan_to_num(X_test.values) @ equal_weights
        out['inverse_ic_vol'] = np.nan_to_num(X_test.values) @ inv_vol_weights
        out['bayesian_ic_weight'] = np.nan_to_num(X_test.values) @ bma_weights

        for name, model in models.items():
            model.fit(X_train, y_train)
            out[name] = model.predict(X_test)

        predictions.append(out)

    return pd.concat(predictions, ignore_index=True)


predictions = fit_predict_walk_forward(panel)
combo_cols = ['equal_weight', 'inverse_ic_vol', 'bayesian_ic_weight', 'ols', 'ridge']
predictions.to_csv(PROCESSED_DIR / 'forecast_combination_predictions_v1.csv', index=False)
predictions.head()

## 6. Evaluation

In [ ]:
def evaluate_forecast(data: pd.DataFrame, forecast_col: str) -> dict:
    ic = data.groupby('date').apply(
        lambda g: g[forecast_col].corr(g['target_next_return'], method='spearman') if len(g) >= 8 else np.nan
    )
    long_short = data.groupby('date').apply(
        lambda g: g.loc[g[forecast_col].rank(pct=True) >= 0.8, 'target_next_return'].mean()
        - g.loc[g[forecast_col].rank(pct=True) <= 0.2, 'target_next_return'].mean()
    )
    return {
        'mean_ic': ic.mean(),
        'ic_vol': ic.std(),
        'ic_ir': ic.mean() / ic.std(),
        'avg_top_minus_bottom_return': long_short.mean(),
        'hit_rate_positive_spread': (long_short > 0).mean(),
    }


evaluation = pd.DataFrame([evaluate_forecast(predictions, col) | {'method': col} for col in combo_cols])
evaluation = evaluation[['method', 'mean_ic', 'ic_vol', 'ic_ir', 'avg_top_minus_bottom_return', 'hit_rate_positive_spread']]
display(evaluation.sort_values('mean_ic', ascending=False))

ic_by_method = pd.DataFrame({
    col: predictions.groupby('date').apply(lambda g: g[col].corr(g['target_next_return'], method='spearman'))
    for col in combo_cols
})

plt.figure(figsize=(10, 5))
ic_by_method.rolling(6).mean().plot(ax=plt.gca())
plt.title('Six-Month Rolling IC by Combination Method')
plt.ylabel('Spearman IC')
plt.show()

## 7. Bias Controls and Next Steps

- Look-ahead bias: signals are shifted so the target is next-month return.
- Shrinkage: ridge regression penalizes extreme combination weights.
- Recency bias: walk-forward training uses a fixed trailing window instead of full-history fitting.
- Survivorship bias remains: this free-data universe only includes current large-cap names.
- V2 should add Project 1 earnings signals, sector neutralization, FRED macro regimes, and delisted names through CRSP.